# Laboratorium 6 - Ćwiczenie 6.1

Implementacja detektora narożników Harrisa zgodnie z instrukcją:
- pochodne Sobela (`Ix`, `Iy`),
- macierz autokorelacji i odpowiedź `H`,
- normalizacja `H` do `[0, 1]`,
- lokalne maksima (`size=7`) i próg,
- wizualizacja punktów na parach obrazów `fontanna` i `budynek`.


In [1]:
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np


In [2]:
def harris_response(
    image_gray: np.ndarray,
    sobel_ksize: int = 7,
    gauss_ksize: int = 7,
    k: float = 0.05,
) -> np.ndarray:
    """Compute normalized Harris response H in range [0, 1]."""
    image_f = image_gray.astype(np.float32)

    ix = cv2.Sobel(image_f, cv2.CV_32F, 1, 0, ksize=sobel_ksize)
    iy = cv2.Sobel(image_f, cv2.CV_32F, 0, 1, ksize=sobel_ksize)

    ixx = cv2.GaussianBlur(ix * ix, (gauss_ksize, gauss_ksize), sigmaX=0)
    iyy = cv2.GaussianBlur(iy * iy, (gauss_ksize, gauss_ksize), sigmaX=0)
    ixy = cv2.GaussianBlur(ix * iy, (gauss_ksize, gauss_ksize), sigmaX=0)

    det_m = ixx * iyy - ixy * ixy
    trace_m = ixx + iyy
    h = det_m - k * (trace_m * trace_m)

    h_min = float(h.min())
    h_max = float(h.max())
    if h_max == h_min:
        return np.zeros_like(h, dtype=np.float32)
    return ((h - h_min) / (h_max - h_min)).astype(np.float32)


def find_max(image: np.ndarray, size: int, threshold: float) -> tuple[np.ndarray, np.ndarray]:
    """Return (ys, xs) for local maxima above threshold."""
    kernel = np.ones((size, size), dtype=np.uint8)
    data_max = cv2.dilate(image, kernel)
    maxima = (image == data_max) & (image > threshold)
    return np.nonzero(maxima)


def draw_pair_with_stars(
    left_img: np.ndarray,
    left_pts: tuple[np.ndarray, np.ndarray],
    left_title: str,
    right_img: np.ndarray,
    right_pts: tuple[np.ndarray, np.ndarray],
    right_title: str,
) -> None:
    ys_l, xs_l = left_pts
    ys_r, xs_r = right_pts

    fig, axes = plt.subplots(1, 2, figsize=(14, 6))

    axes[0].imshow(left_img, cmap="gray")
    axes[0].plot(xs_l, ys_l, "r*", markersize=4)
    axes[0].set_title(f"{left_title} | points: {len(xs_l)}")
    axes[0].axis("off")

    axes[1].imshow(right_img, cmap="gray")
    axes[1].plot(xs_r, ys_r, "r*", markersize=4)
    axes[1].set_title(f"{right_title} | points: {len(xs_r)}")
    axes[1].axis("off")

    fig.tight_layout()
    plt.show()


In [ ]:
def process_pair(
    left_path: Path,
    right_path: Path,
    sobel_size: int = 7,
    gauss_size: int = 7,
    max_size: int = 7,
    threshold: float = 0.6,
) -> tuple[tuple[np.ndarray, np.ndarray], tuple[np.ndarray, np.ndarray]]:
    left = cv2.imread(str(left_path), cv2.IMREAD_GRAYSCALE)
    right = cv2.imread(str(right_path), cv2.IMREAD_GRAYSCALE)

    if left is None or right is None:
        raise FileNotFoundError(f"Cannot read one of images: {left_path}, {right_path}")

    h_left = harris_response(left, sobel_ksize=sobel_size, gauss_ksize=gauss_size)
    h_right = harris_response(right, sobel_ksize=sobel_size, gauss_ksize=gauss_size)

    pts_left = find_max(h_left, size=max_size, threshold=threshold)
    pts_right = find_max(h_right, size=max_size, threshold=threshold)

    print(f"{left_path.name}: {len(pts_left[0])} points")
    print(f"{right_path.name}: {len(pts_right[0])} points")

    draw_pair_with_stars(left, pts_left, left_path.name, right, pts_right, right_path.name)

    return pts_left, pts_right


DATA_DIR = Path("data")

fontanna_pts = process_pair(DATA_DIR / "fontanna1.jpg", DATA_DIR / "fontanna2.jpg")
budynek_pts = process_pair(DATA_DIR / "budynek1.jpg", DATA_DIR / "budynek2.jpg")


[ WARN:0@0.173] global loadsave.cpp:278 findDecoder imread_('lab06/data/fontanna1.jpg'): can't open/read file: check file path/integrity
[ WARN:0@0.173] global loadsave.cpp:278 findDecoder imread_('lab06/data/fontanna2.jpg'): can't open/read file: check file path/integrity


FileNotFoundError: Cannot read one of images: lab06/data/fontanna1.jpg, lab06/data/fontanna2.jpg